# Phase 5.5: Docking the leads into DPP-IV

The 412 calibrated discoveries from phase 5.3 are split into two tiers, a short
Tier 1 and a longer Tier 2, and a handful of each are docked into DPP-IV against
two known inhibitors and three model-confident non-ADPs. The controls are what
make the lead scores readable: without a decoy floor a favourable HADDOCK score
means nothing on its own.

Tiering comes first because it decides which peptides are docked at all. Tier 1
is short peptides at or below 10 residues, the size the DPP-IV pocket takes, kept
whatever the safety tools say about them, since both tools over-flag short
sequences and section 2 measures that. Tier 2 is longer peptides that pass both
safety filters outright.

This is a notebook rather than a script because HADDOCK runs on a web server, so
the pipeline stops halfway and waits for the run archives to come back. Section 5
is that pause.

    INPUTS   screening/screening_discovery.csv    (412 discoveries, from phase 5.3)
             screening/toxinpred2_discovery.csv   (ToxinPred2 scores, from phase 5.4)
             screening/algpred2_safe.csv          (AlgPred calls, from phase 5.4)
             screening/screening_candidates.csv   (3,596 candidates, from phase 5.1)
             data/dataset_split.csv
             docking/haddock_runs/*_summary.tgz   (downloaded by hand in section 5)

    OUTPUTS  screening/tiered_discovery.csv        the tiered table
             results/phase5_5_length_artefact.csv  flag rate by length class
             results/phase5_5_screening_funnel.csv stage-by-stage counts
             docking/receptor_DPP4_4A5S.pdb        cleaned receptor (chain A)
             docking/peptides/<id>.pdb             3-model peptide ensembles (chain B)
             docking/docking_manifest.csv          what was docked and why
             docking/haddock_active_residues.txt   active residues for the AIRs
             docking/haddock_scores.csv            per-peptide best cluster
             results/phase5_5_docking.csv          the same, for section 3.4
             results/phase5_5_docking_summary.csv  the group figures quoted in 3.4

    REQUIREMENTS  pip install pandas scipy PeptideBuilder biopython


In [ ]:
# Imports
import csv
import glob
import io
import os
import re
import tarfile
import urllib.request
import warnings
warnings.filterwarnings("ignore")
from statistics import mean, pstdev # population sd, matching haddock's own report

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

import Bio.PDB
import PeptideBuilder as PB
from Bio.Data.IUPACData import protein_letters_3to1

# a notebook kernel starts in the notebook's own folder rather than the repo root,
# so step up until the paths below resolve. everything is then repo-root relative,
# matching the scripts
while not os.path.isdir("screening") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

os.getcwd()


## 1. Tier the discoveries

Tier 1 is every discovery of 10 residues or fewer. Their toxicity and
allergenicity scores are carried through and soft-flagged rather than filtered
on, because both tools flag short peptides at a higher rate regardless of what
the peptide is, which section 2 tests. Tier 2 is everything longer that passes
both filters outright, so the two tiers are selected on different rules by
design.


In [ ]:
DISCOVERY = "screening/screening_discovery.csv" # 412 calibrated discoveries
TOXPRED = "screening/toxinpred2_discovery.csv" # ToxinPred2 scores, from phase 5.4
ALGPRED = "screening/algpred2_safe.csv" # AlgPred calls, from phase 5.4
CANDIDATES = "screening/screening_candidates.csv" # the 3,596 before scoring
SPLIT = "data/dataset_split.csv"

OUT_TIERED = "screening/tiered_discovery.csv"
OUT_LEN = "results/phase5_5_length_artefact.csv"
OUT_FUNNEL = "results/phase5_5_screening_funnel.csv"

THRESHOLD = 0.90 # mirrors DISCOVERY_THRESHOLD in phase5.3, used here only as a label
SHORT_LEN = 10 # Tier 1 = short peptides at/below this length (ideal DPP-IV pocket size)
TOX_CUT = 0.6 # ToxinPred2 non-toxic threshold

# keep_default_na=False, or the dipeptide NA would parse as a missing value
disc = pd.read_csv(DISCOVERY, keep_default_na=False)
tox = pd.read_csv(TOXPRED, keep_default_na=False).rename(columns={"ML_Score": "tox_score"})
m = disc.merge(tox[["Sequence", "tox_score"]], left_on="sequence", right_on="Sequence",
               how="left").drop(columns="Sequence")

# read in AlgPred allergen predictions and merge
alg = pd.read_csv(ALGPRED, keep_default_na=False)
alg.columns = [c.strip() for c in alg.columns] # "ML Score" has a space
alg = alg.rename(columns={"Subject": "peptide_id", "ML Score": "alg_score", "Prediction": "alg_pred"})
m = m.merge(alg[["peptide_id", "alg_score", "alg_pred"]], on="peptide_id", how="left")

# assign tiers
m["tier"], m["flag"] = "", ""
t1 = m.length <= SHORT_LEN # Tier 1: short, high-consensus
m.loc[t1, ["tier", "flag"]] = ["1", "soft-flag: short-length safety artefact"]
t2 = (~t1) & (m.tox_score < TOX_CUT) & (m.alg_pred == "Non-Allergen") # Tier 2: longer, hard-filtered
m.loc[t2, ["tier", "flag"]] = ["2", "passed hard toxicity + allergen filters"]

print(f"{len(m)} discoveries -> tier 1: {int(t1.sum())} | tier 2: {int(t2.sum())}")
m.head()


## 2. Length-artefact control

The reason Tier 1 is soft-flagged rather than filtered. Both safety tools score
on amino-acid composition, which is noisier over a handful of residues, so a
short peptide is more likely to be flagged whatever it is. Fisher's exact test
compares the flag rate above and below the same 10-residue cut used for the
tiers.

The left merges above leave NaN for anything the tools never scored, so each
tool selects on `isin` rather than on `!= ""`.


In [ ]:
rows, short = [], m.length <= SHORT_LEN
for tool, scored, flag in [("toxinpred2", m.tox_score.notna(), m.tox_score >= TOX_CUT),
                           ("algpred2", m.alg_pred.isin(["Allergen", "Non-Allergen"]),
                            m.alg_pred == "Allergen")]:
    tab = [[int((scored & short & flag).sum()),  int((scored & short & ~flag).sum())],
           [int((scored & ~short & flag).sum()), int((scored & ~short & ~flag).sum())]]
    p = fisher_exact(tab)[1]
    for i, sel in enumerate([short, ~short]):
        n = int((scored & sel).sum())
        rows.append({"tool": tool, "length_class": ["<=10", ">10"][i], "n_scored": n,
                     "n_flagged": tab[i][0], "pct_flagged": round(100 * tab[i][0] / n, 1),
                     "fisher_p": float(f"{p:.3g}")})

length_artefact = pd.DataFrame(rows)
length_artefact.to_csv(OUT_LEN, index=False)
length_artefact


## 3. Screening funnel and the tiered table

The stage-by-stage counts behind figure 5, from the source proteins down to the
two tiers. `overlap_with_split` is the check that nothing recovered by the screen
was already in the training or test data.

The tiered table is sorted by tier and then by consensus. `kind="stable"` is on
the sort because 43 of the 243 tiered rows share a tier and consensus value, so
without it their order is left to the sort algorithm rather than to the input.


In [ ]:
cand = pd.read_csv(CANDIDATES, keep_default_na=False)
split = pd.read_csv(SPLIT, keep_default_na=False)
n_prot = len({p for v in cand.sources for p in str(v).split(";")})

funnel = [
    ("source_proteins", n_prot, "UniProt entries contributing candidates"),
    ("novel_candidates", len(cand), "unique digestion fragments after novelty filter"),
    ("overlap_with_split", len(set(cand.sequence) & set(split.Sequence)),
     "exact matches against train plus test"),
    ("discoveries", len(disc), f"consensus >= {THRESHOLD:.2f}"),
    ("non_toxic", int((m.tox_score < TOX_CUT).sum()), f"ToxinPred2 score < {TOX_CUT}"),
    ("non_toxic_non_allergen",
     int(((m.tox_score < TOX_CUT) & (m.alg_pred == "Non-Allergen")).sum()),
     "AlgPred 2.0 Non-Allergen"),
    ("tier1", int(t1.sum()), f"discoveries of {SHORT_LEN} aa or fewer"),
    ("tier2", int(t2.sum()), "longer discoveries passing both filters"),
]
pd.DataFrame(funnel, columns=["stage", "n", "detail"]).to_csv(OUT_FUNNEL, index=False)

# save the tiered table
tiered = m[m.tier != ""].sort_values(["tier", "consensus"], ascending=[True, False],
                                     kind="stable").reset_index(drop=True)
cols = ["tier", "peptide_id", "sequence", "length", "sources", "consensus", "tox_score", "alg_score", "flag"]
tiered[cols].to_csv(OUT_TIERED, index=False)

print(f"{len(tiered)} tiered peptides -> {OUT_TIERED}")
tiered[cols].head(10)


## 4. Build the HADDOCK inputs

Eight peptides go to docking: the top Tier-1 lead, two Tier-2 leads of different
lengths, two known DPP-IV inhibitors as positive controls and three
model-confident non-ADPs as a noise floor. Each is docked separately and judged
against the controls, not against an absolute score.

Each peptide is handed over as a three-conformation ensemble rather than one
guessed shape, since HADDOCK refines flexibly from whatever it is given and a
single starting conformation biases where it can end up. The receptor is chain A
of 4A5S with waters, sugars, ions and the co-crystal ligand stripped out, and the
active site is whatever sits within 5 A of that ligand.

**This section needs network access**, since it pulls 4A5S from RCSB.


In [ ]:
OUT_DIR = "docking"
PDB_ID, CHAIN, LIGAND = "4A5S", "A", "N7F"
SITE_CUTOFF = 5.0 # A: receptor residues within this of the ligand = active site
PEP_CHAIN = "B"
CONF = {"ext": (-139, 135), "helix": (-57, -47), "ppii": (-75, 145)} # backbone phi/psi per conformation

# role = tier1 / tier2 / pos_control / neg_control
PEPTIDES = [
    ("LEAD_T1_FVAPFPEVF", "FVAPFPEVF", "tier1"), # top short lead (9 aa)
    ("LEAD_T2_CAND3348", "ALPMHIRLSFNPTQLEEQCHI", "tier2"), # Tier-2 lead (21 aa)
    ("LEAD_T2_CAND3554", "TNDTPMIGTLAGANSLLNALPEEVIQHTFNLK", "tier2"), # longer Tier-2 lead (32 aa) for short-vs-long
    ("POS_DiprotinA", "IPI", "pos_control"), # textbook DPP-IV inhibitor
    ("POS_IPAVF", "IPAVF", "pos_control"), # known inhibitor the screen recovered
    ("NEG_hard_1", "RRRLNKHEE", "neg_control"), # model-confident non-ADP (cons ~0.004) for noise floor
    ("NEG_hard_2", "SSKDRLRR", "neg_control"), # model-confident non-ADP
    ("NEG_hard_3", "GRADR", "neg_control"), # model-confident non-ADP
]


In [ ]:
def fetch_pdb(pdb_id):
    # Pull the raw PDB straight from RCSB and hand back a list of lines.
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    return urllib.request.urlopen(url, timeout=30).read().decode().splitlines()


def clean_receptor(raw, chain, out_path):
    """Keep only chain `chain` protein atoms (drop waters, sugars, ligand, ions, other chains).
    Also collapse alternate conformations: keep the first form of each atom and blank the altLoc
    column - otherwise HADDOCK rejects the file for having 'multiple forms' of a residue."""
    kept, seen = [], set()
    for l in raw:
        if not (l.startswith("ATOM") and l[21] == chain):
            continue
        key = (l[12:16], l[22:27]) # atom name + residue (resSeq + insertion code)
        if key in seen: # an alternate conformer of an atom we already kept -> skip
            continue
        seen.add(key)
        kept.append(l[:16] + " " + l[17:]) # blank col 17 (altLoc) so no stale conformer flag remains
    with open(out_path, "w") as fh:
        fh.write("\n".join(kept) + "\nTER\nEND\n")


def active_site(raw, chain, ligand, cutoff):
    """Receptor residues within `cutoff` A of the co-crystal ligand -> the binding site."""
    # Grab the ligand's atom coordinates (cols 31-54 = x, y, z). these mark the pocket
    lig = np.array([(float(l[30:38]), float(l[38:46]), float(l[46:54]))
                    for l in raw
                    if l.startswith("HETATM") and l[17:20].strip() == ligand and l[21] == chain])

    # Any protein residue with an atom closer than `cutoff` to any ligand atom is in the site.
    site = set()
    for l in raw:
        if l.startswith("ATOM") and l[21] == chain:
            xyz = np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
            if lig.size and np.sqrt(((lig - xyz) ** 2).sum(1)).min() < cutoff:
                site.add(int(l[22:26])) # col 23-26 = residue number
    return sorted(site)


def build_ensemble(seq, chain, out_path):
    """3-conformation peptide ensemble (extended/helix/PPII) as a multi-model PDB.

    HADDOCK docks the peptide flexibly from a few starting shapes, so hand it one
    extended, one helical and one polyproline-II conformation rather than guessing one."""
    models = []
    for i, (phi, psi) in enumerate(CONF.values(), 1):
        # build the backbone at fixed phi/psi -> n residues have n-1 angles between them
        st = PB.make_structure(seq, [phi] * (len(seq) - 1), [psi] * (len(seq) - 1))

        # PeptideBuilder writes chain A; rewrite col 22 to peptide chain (B) so it
        # doesn't clash with the receptor's chain A.
        buf = io.StringIO(); w = Bio.PDB.PDBIO(); w.set_structure(st); w.save(buf)
        body = [l[:21] + chain + l[22:] if l.startswith(("ATOM", "TER")) and len(l) >= 22 else l
                for l in buf.getvalue().splitlines() if l.startswith(("ATOM", "TER"))]

        # wrap each conformation as MODEL..ENDMDL so HADDOCK reads them as one ensemble
        models.append(f"MODEL     {i}\n" + "\n".join(body) + "\nENDMDL")
    with open(out_path, "w") as fh:
        fh.write("\n".join(models) + "\nEND\n")


In [ ]:
# prep the docking inputs: cleaned receptor, active site, peptide ensembles, manifest
os.makedirs(f"{OUT_DIR}/peptides", exist_ok=True)
raw = fetch_pdb(PDB_ID)

clean_receptor(raw, CHAIN, f"{OUT_DIR}/receptor_DPP4_{PDB_ID}.pdb")
site = active_site(raw, CHAIN, LIGAND, SITE_CUTOFF)

# write the peptide ensembles and the docking manifest
manifest = pd.DataFrame(PEPTIDES, columns=["peptide_id", "sequence", "role"])
manifest["length"] = manifest.sequence.str.len()
manifest.to_csv(f"{OUT_DIR}/docking_manifest.csv", index=False)
for _, r in manifest.iterrows():
    build_ensemble(r.sequence, PEP_CHAIN, f"{OUT_DIR}/peptides/{r.peptide_id}.pdb")

# write the HADDOCK active residues (ambiguous interaction restraints) file
with open(f"{OUT_DIR}/haddock_active_residues.txt", "w") as fh:
    fh.write("# HADDOCK active residues (ambiguous interaction restraints)\n")
    fh.write(f"receptor (chain {CHAIN}) active: {','.join(map(str, site))}\n")
    fh.write(f"peptide  (chain {PEP_CHAIN}) active: all residues (1..N per peptide)\n")

print(f"receptor -> {OUT_DIR}/receptor_DPP4_{PDB_ID}.pdb")
print(f"active site ({len(site)} residues): {site}")
print(f"{len(manifest)} peptide ensembles -> {OUT_DIR}/peptides/")
manifest


## 5. Dock on the HADDOCK web server

**Manual step.** HADDOCK 2.4 runs on the WeNMR portal, so the eight jobs are
submitted by hand and their results downloaded before the next section can read
anything.

https://wenmr.science.uu.nl/haddock2.4/

For each peptide, submit `docking/receptor_DPP4_4A5S.pdb` as molecule 1 and
`docking/peptides/<id>.pdb` as molecule 2, with the active residues from
`docking/haddock_active_residues.txt` on each side. The peptide ensemble is a
multi-model PDB, so allow all three conformations rather than only the first.
The full protocol, including every non-default setting, is in table 5.

Download each finished run as its summary archive and put all eight in
`docking/haddock_runs/`, keeping the portal's filename. The job number in front
of the name is what carries the run id into the results table:

    docking/haddock_runs/734078-DPP4_FVAPFPEVF_summary.tgz
    docking/haddock_runs/734086-DPP4_CAND3348_summary.tgz
    ...

Run the next cells once the archives are in place.


## 6. Read the runs back

HADDOCK returns a folder of model PDBs per peptide but no single table to compare
them, and it does not store the final score either. So each model's score is
rebuilt from the energies in its own PDB header, averaged across the models in a
pose cluster, and the best cluster kept.

The weights are the water-stage ones, since every archived model is water-refined
(the filenames end `w.pdb`) and that stage downweights electrostatics to account
for solvent screening. Buried surface area is read but not scored, because the
water-refinement function does not include it. More negative is a tighter
predicted binder.

https://www.bonvinlab.org/software/haddock2.4/scoring/


In [ ]:
ARCHIVE_DIR = "docking/haddock_runs" # the downloaded summary archives from section 5
OUT_SCORES = "docking/haddock_scores.csv"
OUT_RESULTS = "results/phase5_5_docking.csv"
OUT_SUMMARY = "results/phase5_5_docking_summary.csv"

#   vdw shape packing, elec charge attraction and repulsion, desolv rewards burying
#   water-repelling surface, air restraint-violation penalty rather than a binding energy.
W_VDW, W_ELEC, W_DESOLV, W_AIR = 1.0, 0.2, 1.0, 0.1

# job name to what role the peptide plays here. the sequence itself is read from the pdb,
# so this only records the experiment: a discovery lead, a known binder, or a known non-binder.
# the keys are the HADDOCK job names, which is why they differ from the ids in PEPTIDES above
ROLES = {
    "DPP4_FVAPFPEVF": "Tier-1 lead", "DPP4_CAND3554": "Tier-2 lead",
    "DPP4_CAND3348": "Tier-2 lead", "DPP4_DiprotinA": "+ve control",
    "DPP4_IPAVF": "+ve control", "DPP4_NEGhard1":  "-ve control",
    "DPP4_NEGhard2": "-ve control", "DPP4_NEGhard3": "-ve control",
}

# three-letter to one-letter amino-acid codes, for spelling the peptide out of the pdb.
# biopython keys these capitalised ("Ala"), the pdb writes them upper case ("ALA")
AA3 = {k.upper(): v for k, v in protein_letters_3to1.items()}

# every model is named "clusterN_M.pdb": N = which cluster, M = its 1-4 rank inside that cluster
CL_RE = re.compile(r"cluster(\d+)_(\d+)\.pdb$")


In [ ]:
def parse_rep(text):
    """one model pdb -> (haddock score, its energy breakdown, peptide sequence).

    haddock writes the energies as plain-text REMARK lines at the top of each pdb, so
    scan the text for them and add up the score (the score isn't stored directly).
    """
    vdw = elec = air = desolv = bsa = None # define as none to detect a half-written model
    chain_ca, resn = {}, {} # per-chain Ca counts + residue names

    for ln in text.splitlines():
        # energies come as one fixed-order, comma-separated line - strip the REMARK energies:
        # label, split on commas and take the three we need by position:
        #   0=total 1=bonds 2=angles 3=improper 4=dihe  5=vdw  6=elec  7=air  8+=other restraints (unused)
        if ln.startswith("REMARK energies:"):
            v = ln.split(":", 1)[1].split(",")
            vdw, elec, air = float(v[5]), float(v[6]), float(v[7])

        # the desolvation energy and buried surface area are each on their own line
        elif ln.startswith("REMARK Desolvation energy:"):
            desolv = float(ln.split(":", 1)[1])

        # buried surface area is also on its own line, but don't use it in the score
        elif ln.startswith("REMARK buried surface area:"):
            bsa = float(ln.split(":", 1)[1])

        # every residue has one "CA" backbone atom. Counting them per chain gives chain
        # length and their residue names, read in order, spell the sequence
        elif ln.startswith("ATOM") and ln[12:16].strip() == "CA":
            ch = ln[21] # chain id sits at column 22 of a pdb line
            chain_ca[ch] = chain_ca.get(ch, 0) + 1 # +1 because the CA line is always present for every residue

            # residue name is in columns 18-20, residue number in 23-26. store them per chain
            resn.setdefault(ch, {})[int(ln[22:26])] = ln[17:20].strip()

    # a cut-off download can leave a half-written model,
    # so if any of the energies are missing or no CA atoms were found, skip it
    if None in (vdw, elec, air, desolv, bsa) or not chain_ca:
        return None

    # haddock's score is a weighted sum of the four energies, with the weights defined above
    score = W_VDW*vdw + W_ELEC*elec + W_DESOLV*desolv + W_AIR*air
    lig = min(chain_ca, key=chain_ca.get) # two chains: DPP-IV (big) + peptide (small)

    # spell the peptide out of its residue names, in N- to C-terminus order.
    # if a residue is missing, use "X" to mark it.
    seq = "".join(AA3.get(resn[lig][n], "X") for n in sorted(resn[lig])) # N- to C-terminus
    return score, {"vdw":vdw, "elec":elec, "desolv":desolv, "air":air, "bsa":bsa}, seq


def parse_job(tgz):
    """Read one peptide's docking archive and return the best cluster summary.

    Returns None if the archive contains no usable cluster files.
    """
    clusters = {} # cluster number -> list of (score, energy_breakdown)
    seq = None # peptide sequence, recovered from the first valid model

    # Read the .tgz directly instead of unpacking it first
    with tarfile.open(tgz) as tar:
        for mem in tar.getmembers():

            # Only process files that look like HADDOCK cluster PDBs
            hit = CL_RE.search(os.path.basename(mem.name))
            if not hit:
                continue

            # Parse one model PDB from the archive member
            rep = parse_rep(tar.extractfile(mem).read().decode())
            if rep is None:
                # A missing REMARK/ATOM section usually means the file is incomplete
                continue

            score, comp, s = rep
            cl = int(hit.group(1))

            # Store this model under its cluster number
            clusters.setdefault(cl, []).append((score, comp))

            # Save the peptide sequence the first time we see it
            seq = seq or s

    if not clusters:
        return None

    # Collapse each cluster's models into one summary row
    # score = mean HADDOCK score across models in the cluster
    # sd = spread of those scores
    # n = how many models were found in that cluster
    # other fields = mean of each energy term across models
    stats = {}
    for cl, reps in clusters.items():
        scores = [score for score, _ in reps]
        stats[cl] = {
            "score": mean(scores),
            "sd": pstdev(scores),
            "n": len(scores),
            **{k: mean(comp[k] for _, comp in reps) for k in reps[0][1]},
        }

    # Pick the cluster with the lowest score
    # More negative = better predicted binder
    best = min(stats, key=lambda c: stats[c]["score"])

    # Compare the best cluster against the rest
    means = [stats[c]["score"] for c in stats]
    z = ((stats[best]["score"] - mean(means)) / pstdev(means)) if len(means) > 1 else None # calculate z-score only if more than one cluster exists

    return {
        "peptide": seq,
        "n_clusters": len(stats),
        "best_cl": best,
        "z": z,
        **stats[best],
    }


In [ ]:
def weighted_terms(rows):
    """Add each energy term's weighted contribution to the score, plus the share
    of the favourable total carried by electrostatics."""
    for r in rows:
        r["vdw_contrib"]    = round(W_VDW    * r["vdw"], 2)
        r["elec_contrib"]   = round(W_ELEC   * r["elec"], 2)
        r["desolv_contrib"] = round(W_DESOLV * r["desolv"], 2)
        r["air_contrib"]    = round(W_AIR    * r["air"], 2)
        fav = sum(v for v in (r["vdw_contrib"], r["elec_contrib"],
                              r["desolv_contrib"], r["air_contrib"]) if v < 0)
        r["elec_pct_of_favourable"] = round(100 * r["elec_contrib"] / fav, 1)
        r["desolv_favourable"] = r["desolv"] < 0


# write out the results table and the group figures quoted in results section 3.4
def write_results(rows):
    """One row per docked peptide, plus the group figures quoted in Section 3.4."""
    os.makedirs(os.path.dirname(OUT_RESULTS) or ".", exist_ok=True)
    cols = ["role", "peptide", "job_id", "score", "sd", "n", "best_cl", "n_clusters", "z",
            "vdw", "elec", "desolv", "air", "bsa",
            "vdw_contrib", "elec_contrib", "desolv_contrib", "air_contrib",
            "elec_pct_of_favourable", "desolv_favourable"]
    with open(OUT_RESULTS, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: (round(v, 4) if isinstance(v, float) else v)
                        for k, v in r.items() if k in cols})

    decoys = [r for r in rows if r["role"] == "-ve control"]
    genuine = [r for r in rows if r["role"] in ("Tier-1 lead", "+ve control")]
    rng = lambda g, k: f"{min(r[k] for r in g):.1f} to {max(r[k] for r in g):.1f}"
    summary = [
        ("n_docked", len(rows)),
        ("n_decoys", len(decoys)),
        ("best_score_peptide", rows[0]["peptide"]),
        ("best_score", round(rows[0]["score"], 1)),
        ("best_score_sd", round(rows[0]["sd"], 1)),
        ("best_score_z", round(rows[0]["z"], 2)),
        ("decoy_score_range", rng(decoys, "score")),
        ("decoy_elec_range", rng(decoys, "elec")),
        ("decoy_desolv_range", rng(decoys, "desolv")),
        ("decoy_elec_pct_range", rng(decoys, "elec_pct_of_favourable")),
        ("genuine_elec_pct_range", rng(genuine, "elec_pct_of_favourable")),
        ("n_desolv_unfavourable", sum(1 for r in rows if not r["desolv_favourable"])),
        ("leads_desolv_unfavourable",
         ";".join(r["peptide"] for r in rows
                  if not r["desolv_favourable"] and r["role"].endswith("lead"))),
    ]
    with open(OUT_SUMMARY, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["quantity", "value"])
        w.writerows(summary)


In [ ]:
rows = []

# Find every downloaded HADDOCK summary archive.
for tgz in sorted(glob.glob(os.path.join(ARCHIVE_DIR, "*_summary.tgz"))):
    base = os.path.basename(tgz)

    # Extract the job ID and job name from the filename.
    # Example: 12345-DPP4_IPAVF_summary.tgz
    jid = base.split("-", 1)[0]
    name = base.split("-", 1)[1].rsplit("_summary", 1)[0]

    # Parse the archive into one summary row.
    r = parse_job(tgz)
    if r is None:
        continue

    # Add job metadata to the parsed docking result.
    rows.append({"job_id": jid, "job": name, "role": ROLES.get(name, "?"), **r})

assert rows, f"no usable archives in {ARCHIVE_DIR}, see the step above"

# Sort from best binder to worst.
rows.sort(key=lambda r: r["score"])
weighted_terms(rows)
write_results(rows)

# Write the final results table.
os.makedirs(os.path.dirname(OUT_SCORES) or ".", exist_ok=True)
cols = ["role", "job_id", "job", "peptide", "score", "sd", "n", "best_cl", "n_clusters",
    "z", "vdw", "elec", "desolv", "air", "bsa",
    "vdw_contrib", "elec_contrib", "desolv_contrib", "air_contrib",
    "elec_pct_of_favourable", "desolv_favourable"]
with open(OUT_SCORES, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)

print(f"{len(rows)} runs parsed -> {OUT_SCORES}")
pd.read_csv(OUT_SCORES)[["role", "job", "peptide", "score", "sd", "z", "elec_pct_of_favourable"]]
